In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

# --- 1. DATA PREPARATION (BINARY) ---
def is_prime(n):
    if n < 2: return 0
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0: return 0
    return 1

# Convert number to binary list, e.g., 5 -> [0...0, 1, 0, 1]
# We use 14 bits because 2^14 = 16384 (covers up to 10,000)
def num_to_binary(n, max_bits=14):
    binary_str = format(n, f'0{max_bits}b')
    return [int(b) for b in binary_str]

numbers = range(1, 10001)
X = np.array([num_to_binary(n) for n in numbers]) # Shape: (10000, 14)
y = np.array([is_prime(n) for n in numbers])

# --- 2. BUILD MODEL ---
# Now the input shape is 14 (one neuron for each bit)
model = Sequential([
    Input(shape=(14,)),
    Dense(128, activation='relu'), # More neurons to capture logic gates
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# --- 3. TRAIN (CORRECTED) ---
from sklearn.utils.class_weight import compute_class_weight

# ERROR FIX: classes must be a numpy array, so we use np.unique(y)
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)

# Map the weights to the dictionary: {0: weight_for_non_prime, 1: weight_for_prime}
class_weights = dict(zip(np.unique(y), weights))

print(f"Class Weights Computed: {class_weights}")
# You will likely see something like {0: 0.56, 1: 4.06}
# This means the model will treat every Prime number mistake as ~4x worse than a normal mistake.

print("Starting Training with Binary Inputs...")
model.fit(X, y, epochs=20, batch_size=32, verbose=1, class_weight=class_weights)
print("Training Complete.")

# --- 4. PREDICT ---
test_nums = [17, 20, 997, 4000]
X_test = np.array([num_to_binary(n) for n in test_nums])
preds = model.predict(X_test)

for num, pred in zip(test_nums, preds):
    status = "Prime" if pred > 0.5 else "Not Prime"
    # Print the raw probability to see how sure it is
    print(f"Number: {num:<5} | Prediction: {status} ({pred[0]*100:.1f}%)")

Class Weights Computed: {np.int64(0): np.float64(0.5700604264051989), np.int64(1): np.float64(4.068348250610252)}
Starting Training with Binary Inputs...
Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5703 - loss: 0.5670
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6307 - loss: 0.4422
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6097 - loss: 0.4446
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6209 - loss: 0.4421
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6199 - loss: 0.4450
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6275 - loss: 0.4368
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6280 - loss: 0.4301
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6264 - loss: 0.4309
Epoch 9/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6193 - loss: 0.4420
Epoch 10/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6328 